<div style="text-align: center;">
    <h1 style="color: #FF6347;">Self-Guided Lab: Retrieval-Augmented Generation (RAGs)</h1>
</div>

<div style="text-align: center;">
    <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZ3FsdzRveTBrenMxM3VnbDMwaTJxN2NnZm50aGFibXk1NzNnY2Q0MCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/LR5ZBwZHv02lmpVoEU/giphy.gif" alt="NLP Gif" style="width: 300px; height: 150px; object-fit: cover; object-position: center;">
</div>

<h1 style="color: #FF6347;">Data Storage & Retrieval</h1>


<h2 style="color: #FF8C00;">PyPDFLoader</h2>

`PyPDFLoader` is a lightweight Python library designed to streamline the process of loading and parsing PDF documents for text processing tasks. It is particularly useful in Retrieval-Augmented Generation workflows where text extraction from PDFs is required.

- **What Does PyPDFLoader Do?**
  - Extracts text from PDF files, retaining formatting and layout.
  - Simplifies the preprocessing of document-based datasets.
  - Supports efficient and scalable loading of large PDF collections.

- **Key Features:**
  - Compatible with popular NLP libraries and frameworks.
  - Handles multi-page PDFs and embedded images (e.g., OCR-compatible setups).
  - Provides flexible configurations for structured text extraction.

- **Use Cases:**
  - Preparing PDF documents for retrieval-based systems in RAGs.
  - Automating the text extraction pipeline for document analysis.
  - Creating datasets from academic papers, technical manuals, and reports.


In [2]:
!pip install langchain langchain_community pypdf
!pip install termcolor langchain_openai langchain-huggingface sentence-transformers chromadb langchain_chroma tiktoken openai python-dotenv


In [5]:
import os


In [2]:
import warnings
warnings.filterwarnings("ignore")

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings, ChatOpenAI

<h3 style="color: #FF8C00;">Loading the Documents</h3>

In [5]:
# File path for the document

file_path = "../ai-for-everyone.pdf"

<h3 style="color: #FF8C00;">Documents into pages</h3>

The `PyPDFLoader` library allows efficient loading and splitting of PDF documents into smaller, manageable parts for NLP tasks.

This functionality is particularly useful in workflows requiring granular text processing, such as Retrieval-Augmented Generation (RAG).


In [6]:
# Load and split the document
loader = PyPDFLoader(file_path)
pages = loader.load_and_split()
len(pages)

297

<h3 style="color: #FF8C00;">Pages into Chunks</h3>


####  RecursiveCharacterTextSplitter in LangChain

The `RecursiveCharacterTextSplitter` is the **recommended splitter** in LangChain when you want to break down long documents into smaller, semantically meaningful chunks — especially useful in **RAG pipelines**, where clean context chunks lead to better LLM responses.

####  Parameters

| Parameter       | Description                                                                 |
|-----------------|-----------------------------------------------------------------------------|
| `chunk_size`    | The **maximum number of characters** allowed in a chunk (e.g., `1000`).     |
| `chunk_overlap` | The number of **overlapping characters** between consecutive chunks (e.g., `200`). This helps preserve context continuity. |

####  How it works
`RecursiveCharacterTextSplitter` attempts to split the text **intelligently**, trying the following separators in order:
1. Paragraphs (`"\n\n"`)
2. Lines (`"\n"`)
3. Sentences or words (`" "`)
4. Individual characters (as a last resort)

This makes it ideal for handling **natural language documents**, such as PDFs, articles, or long reports, without breaking sentences or paragraphs in awkward ways.



In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(pages)

len(chunks)

1096

####  Alternative: CharacterTextSplitter

`CharacterTextSplitter` is a simpler splitter that breaks text into chunks based **purely on character count**, without trying to preserve any natural language structure.

##### Example:
```python
from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
````

This method is faster and more predictable but may split text in the middle of a sentence or paragraph, which can hurt performance in downstream tasks like retrieval or QA.

---

#### Comparison Table

| Feature                        | RecursiveCharacterTextSplitter | CharacterTextSplitter     |
| ------------------------------ | ------------------------------ | ------------------------- |
| Structure-aware splitting      |  Yes                          |  No                      |
| Preserves sentence/paragraphs  |  Yes                          |  No                      |
| Risk of splitting mid-sentence |  Minimal                     |  High                   |
| Ideal for RAG/document QA      |  Highly recommended           |  Only if structured text |
| Performance speed              |  Slightly slower             |  Faster                  |

---

#### Recommendation

Use `RecursiveCharacterTextSplitter` for most real-world document processing tasks, especially when building RAG pipelines or working with structured natural language content like PDFs or articles.

## Best Practices for Choosing Chunk Size in RAG

### Best Practices for Chunk Size in RAG

| Factor                      | Recommendation                                                                                                                                                                                          |
| ---------------------------| ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **LLM context limit**       | Choose a chunk size that lets you retrieve multiple chunks **without exceeding the model’s token limit**. For example, GPT-4o supports 128k tokens, but with GPT-3.5 (16k) or GPT-4 (32k), keep it modest. |
| **Chunk size (in characters)** | Typically: **500–1,000 characters** per chunk → ~75–200 tokens. This fits well for retrieval + prompt without context overflow.                                                                           |
| **Chunk size (in tokens)**  | If using token-based splitter (e.g. `TokenTextSplitter`): aim for **100–300 tokens** per chunk.                                                                                                            |
| **Chunk overlap**           | Use **overlap of 10–30%** (e.g., 100–300 characters or ~50 tokens) to preserve context across chunk boundaries and avoid cutting off important ideas mid-sentence.                                        |
| **Document structure**      | Use **`RecursiveCharacterTextSplitter`** to preserve semantic boundaries (paragraphs, sentences) instead of arbitrary cuts.                                                                                |
| **Task type**               | For **question answering**, smaller chunks (~500–800 chars) reduce noise.<br>For **summarization**, slightly larger chunks (~1000–1500) are OK.                                                          |
| **Embedding model**         | Some models (e.g., `text-embedding-3-large`) can handle long input. But still, smaller chunks give **finer-grained retrieval**, which improves relevance.                                                  |
| **Query type**              | If users ask **very specific questions**, small focused chunks are better. For broader queries, bigger chunks might help.                                                                                  |


### Rule of Thumb

| Use Case                 | Chunk Size      | Overlap |
| ------------------------| --------------- | ------- |
| Factual Q&A              | 500–800 chars   | 100–200 |
| Summarization            | 1000–1500 chars | 200–300 |
| Technical documents      | 400–700 chars   | 100–200 |
| Long reports/books       | 800–1200 chars  | 200–300 |
| Small LLMs (≤16k tokens) | ≤800 chars      | 100–200 |


### Avoid

- Chunks >2000 characters: risks context overflow.
- No overlap: may lose key information between chunks.



<h2 style="color: #FF8C00;">Embeddings</h2>

Embeddings transform text into dense vector representations, capturing semantic meaning and contextual relationships. They are essential for efficient document retrieval and similarity analysis.

- **What are OpenAI Embeddings?**
  - Pre-trained embeddings like `text-embedding-3-large` generate high-quality vector representations for text.
  - Encapsulate semantic relationships in the text, enabling robust NLP applications.

- **Key Features of `text-embedding-3-large`:**
  - Large-scale embedding model optimized for accuracy and versatility.
  - Handles diverse NLP tasks, including retrieval, classification, and clustering.
  - Ideal for applications with high-performance requirements.

- **Benefits:**
  - Reduces the need for extensive custom training.
  - Provides state-of-the-art performance in retrieval-augmented systems.
  - Compatible with RAGs to create powerful context-aware models.


In [1]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [6]:
api_key = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

<h2 style="color: #FF8C00;">ChromaDB</h2>

ChromaDB is a versatile vector database designed for efficiently storing and retrieving embeddings. It integrates seamlessly with embedding models to enable high-performance similarity search and context-based retrieval.

### Workflow Overview:
- **Step 1:** Generate embeddings using a pre-trained model (e.g., OpenAI's `text-embedding-3-large`).
- **Step 2:** Store the embeddings in ChromaDB for efficient retrieval and similarity calculations.
- **Step 3:** Use the stored embeddings to perform searches, matching, or context-based retrieval.

### Key Features of ChromaDB:
- **Scalability:** Handles large-scale datasets with optimized indexing and search capabilities.
- **Speed:** Provides fast and accurate retrieval of embeddings for real-time applications.
- **Integration:** Supports integration with popular frameworks and libraries for embedding generation.

In [14]:
from langchain_chroma import Chroma

In [15]:
db = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db_LAB")
print("ChromaDB created with document embeddings.")

ChromaDB created with document embeddings.


<h1 style="color: #FF6347;">Retrieving Documents</h1>


### Exercice1: Write a user question that someone might ask about your book’s topic or content.

In [18]:
user_question = "is the EU working on developing an AI" # User question
retrieved_docs = db.similarity_search(user_question, k=10) # k is the number of documents to retrieve

In [19]:
# Display top results
for i, doc in enumerate(retrieved_docs[:3]): # Display top 3 results
    print(f"Document {i+1}:\n{doc.page_content[36:1000]}") # Display content

Document 1:
ch to artificial intelligence based  
on three pillars:
• connect and strengthen AI research centres across Europe;
• support the development of an ‘AI-on-demand platform’ that will provide 
access to relevant AI resources in the EU for all users;
• support the development of AI applications in key sectors (European Com-
mission 2018b, 1).
In order to support the development of the AI strategies summarised here, 
the EC established two advisory entities: The High-Level Expert Group on  
AI (HLEG); and the European AI Alliance. The High-Level Expert Group on AI  
is charged with developing proposals for the overall EU’s AI strategy, policy 
and priorities. It comprises 23 members from industry, 19 from academia and  
10 from civil society; and it is further divided into two working groups: one 
on ethics; and one on investment and policy. The second advisory entity, 
the European AI Alliance, is a multi-stakeholder online platform. On the
Document 2:
gree to a continuous dia

<h2 style="color: #FF8C00;">Preparing Content for GenAI</h2>

In [20]:
def _get_document_prompt(docs):
    prompt = "\n"
    for doc in docs:
        prompt += "\nContent:\n"
        prompt += doc.page_content + "\n\n"
    return prompt

In [21]:
# Generate a formatted context from the retrieved documents
formatted_context = _get_document_prompt(retrieved_docs)
print("Context formatted for GPT model.")

Context formatted for GPT model.


<h2 style="color: #FF8C00;">ChatBot Architecture</h2>

### Exercice2: Write a prompt that is relevant and tailored to the content and style of your book.

In [22]:
prompt = f"""

## SYSTEM ROLE

You are a knowledgeable and factual chatbot designed to assist with questions about Artificial Intelligence (AI).

Your answers must be based exclusively on the provided context extracted from official or academic documents.




## USER QUESTION

The user has asked:

"{user_question}"




## CONTEXT

Here is the relevant content from the documents:

'''

{formatted_context}

'''




## GUIDELINES

1. **Accuracy**:

   - Only use the information provided in the `CONTEXT` section.

   - If the answer is not explicitly present, state:
     "The provided context does not contain sufficient information."

   - Focus specifically on whether the European Union is actively developing or supporting AI initiatives.

   - Mention important EU actions, strategies, organizations, and objectives related to AI.




2. **Transparency**:

   - Reference the relevant document sections when possible.

   - Do not speculate or introduce external knowledge.




3. **Clarity**:

   - Use concise, professional, and easy-to-understand language.

   - Format the response in Markdown for readability.

   - Summarize the main AI initiatives using bullet points when appropriate.




## TASK

1. Answer the user's question directly.

2. Summarize the EU's AI strategy and initiatives mentioned in the context.

3. Highlight important institutions, policies, or platforms related to AI development.

4. Provide the response in the following format:




## RESPONSE FORMAT

'''

# [Brief Title of the Answer]

[Clear and concise explanation.]

## Key EU AI Initiatives

- [Initiative 1]
- [Initiative 2]
- [Initiative 3]

**Source**:

• [Document Name or Section]

'''

"""

print("Prompt constructed.")

Prompt constructed.


In [23]:
import openai

### Exercice3: Tune parameters like temperature, and penalties to control how creative, focused, or varied the model's responses are.

In [26]:
client = openai.OpenAI()

model_params = {
    "model": "gpt-4.1-mini",

    # Lower temperature = more factual / deterministic
    "temperature": 0.2,

    # Enough room for structured answers
    "max_tokens": 800,

    # Default balanced nucleus sampling
    "top_p": 1.0,

    # Keep repetition controlled but natural
    "frequency_penalty": 0.1,

    # Avoid hallucinated topic drifting
    "presence_penalty": 0.0
}

<h1 style="color: #FF6347;">Response</h1>


In [27]:
messages = [{'role': 'user', 'content': prompt}]
completion = client.chat.completions.create(messages=messages, **model_params, timeout=120)

In [28]:
answer = completion.choices[0].message.content
print(answer)

'''

# Is the EU Working on Developing AI?

Yes, the European Union is actively working on developing artificial intelligence through a coordinated strategy that involves research, investment, policy-making, and ethical guidelines. The EU aims to foster AI development by connecting research centers, supporting AI resource platforms, and promoting AI applications in key sectors. It also emphasizes trustworthy AI, ethical standards, and regulatory frameworks to ensure safe and responsible AI deployment.

## Key EU AI Initiatives

- **European Approach to AI (Three Pillars):**  
  - Connecting and strengthening AI research centers across Europe  
  - Supporting the development of an ‘AI-on-demand platform’ to provide access to AI resources for all users in the EU  
  - Supporting AI application development in key sectors

- **Advisory Entities:**  
  - High-Level Expert Group on AI (HLEG), which develops proposals on EU AI strategy, ethics, investment, and policy  
  - European AI Allianc

<img src="https://miro.medium.com/v2/resize:fit:824/1*GK56xmDIWtNQAD_jnBIt2g.png" alt="NLP Gif" style="width: 500px">

<h2 style="color: #FF6347;">Cosine Similarity</h2>

**Cosine similarity** is a metric used to measure the alignment or similarity between two vectors, calculated as the cosine of the angle between them. It is the **most common metric used in RAG pipelines** for vector retrieval.. It provides a scale from -1 to 1:

- **-1**: Vectors are completely opposite.
- **0**: Vectors are orthogonal (uncorrelated or unrelated).
- **1**: Vectors are identical.


<img src="https://storage.googleapis.com/lds-media/images/cosine-similarity-vectors.original.jpg" alt="NLP Gif" style="width: 700px">

<h2 style="color: #FF6347;">Keyword Highlighting</h2>

Highlighting important keywords helps users quickly understand the relevance of the retrieved text to their query.

In [29]:
from termcolor import colored

The `highlight_keywords` function is designed to highlight specific keywords within a given text. It replaces each keyword in the text with a highlighted version using the `colored` function from the `termcolor` library.


In [30]:
def highlight_keywords(text, keywords):
    for keyword in keywords:
        text = text.replace(keyword, colored(keyword, 'green', attrs=['bold']))
    return text

### Exercice4: add your keywords

In [33]:
query_keywords = ["EU"] # add your keywords
for i, doc in enumerate(retrieved_docs[:1]):
    snippet = doc.page_content[:200]
    highlighted = highlight_keywords(snippet, query_keywords)
    print(f"Snippet {i+1}:\n{highlighted}\n{'-'*80}")

Snippet 1:
the EC put forward a European approach to artificial intelligence based  
on three pillars:
• connect and strengthen AI research centres across Europe;
• support the development of an ‘AI-on-demand pl
--------------------------------------------------------------------------------


1. `query_keywords` is a list of keywords to be highlighted.
2. The loop iterates over the first document in retrieved_docs.
3. For each document, a snippet of the first 200 characters is extracted.
4. The highlight_keywords function is called to highlight the keywords in the snippet.
5. The highlighted snippet is printed along with a separator line.

<h1 style="color: #FF6347;">Bonus</h1>

**Try loading one of your own PDF books and go through the steps again to explore how the pipeline works with your content**:


in this Bonus test, I do not own a pdf book - however, I decided to run a similar approach of the above RAG locally using Ollama, Gemma3 4b, and an ollama text embedder. 

In [ ]:
# =========================================================
# LOCAL RAG SYSTEM WITH OLLAMA + GEMMA3 4B
# =========================================================
# Requirements:
"""""
!pip install langchain
!pip install langchain-community
!pip install langchain-ollama
!pip install langchain-chroma
!pip install chromadb
!pip install pypdf
"""
# =========================================================


   ---------------------------------------- 2/2 [langchain-ollama]



In [2]:

import os
import warnings

warnings.filterwarnings("ignore")

# =========================================================
# FORCE NVIDIA GPU (Windows)
# =========================================================

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["OLLAMA_ORIGINS"] = "*"

# =========================================================
# IMPORTS
# =========================================================

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_ollama import (
    ChatOllama,
    OllamaEmbeddings
)

from langchain_chroma import Chroma


In [4]:


PDF_PATH = "../ai-for-everyone.pdf"

CHROMA_DB_DIR = "./chroma_db"

EMBED_MODEL = "nomic-embed-text"

LLM_MODEL = "gemma3:4b"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

TOP_K = 4

In [ ]:
#LOAD PDF

print("\nLoading PDF...\n")

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

print(f"Loaded {len(documents)} pages.")


Loading PDF...

Loaded 310 pages.


In [6]:
# =========================================================
# SPLIT DOCUMENTS
# =========================================================

print("\nSplitting text into chunks...\n")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

splits = text_splitter.split_documents(documents)

print(f"Created {len(splits)} chunks.")



Splitting text into chunks...

Created 1096 chunks.


In [7]:
#ollama embeding
print("\nLoading embedding model...\n")

embeddings = OllamaEmbeddings(
    model=EMBED_MODEL
)



Loading embedding model...



In [9]:
# VECTOR DATABASE

print("\nCreating vector database...\n")

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory=CHROMA_DB_DIR
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": TOP_K}
)

print("Vector DB ready.")



Creating vector database...

Vector DB ready.


In [10]:
# =========================================================
# LOAD LOCAL GEMMA MODEL
# =========================================================

print("\nLoading Gemma3 4B...\n")

llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0.2,
    num_predict=512,
)

print("Gemma loaded successfully.")



Loading Gemma3 4B...

Gemma loaded successfully.



 LOCAL RAG CHAT READY 
 Type 'exit' to quit



In [25]:

# =========================================================
# CHAT LOOP
# =========================================================

print("\n=================================================")
print(" LOCAL RAG CHAT READY ")
print(" Type 'exit' to quit")
print("=================================================\n")

while True:

    question = input("You: ")

    if question.lower() == "exit":
        break

    # =====================================================
    # RETRIEVE RELEVANT CHUNKS
    # =====================================================

    retrieved_docs = retriever.invoke(question)

    context_parts = []

    for i, doc in enumerate(retrieved_docs):

        page = doc.metadata.get("page", "Unknown")

        text = doc.page_content

        context_parts.append(
            f"""
    SOURCE {i+1}
    PAGE: {page + 1}

    {text}
    """
        )

    context = "\n\n".join(context_parts)


    # =====================================================
    # PROMPT
    # =====================================================

    prompt = f"""
    You are an advanced document analysis assistant.

    Answer the user's question ONLY using the retrieved text.

    IMPORTANT:
    - Cite the page number(s) used.
    - Format citations like: (Page 3)
    - If multiple pages are used, cite all relevant pages.
    - Do not invent page numbers.

    ================ RETRIEVED DOCUMENTS ================

    {context}

    =====================================================

    QUESTION:
    {question}

    =====================================================

    Provide a precise answer with page citations:
    """

    # =====================================================
    # GENERATE RESPONSE
    # =====================================================

    response = llm.invoke(prompt)

    print("\nAssistant:\n")
    print(response.content)
    print("\n" + "=" * 50 + "\n")


 LOCAL RAG CHAT READY 
 Type 'exit' to quit


Assistant:

Based on the provided text, AI is presented as a means to tackle many fundamental problems of mankind, including fighting the climate crisis, curing cancer, and improving predictive police work (Page 99). Furthermore, AI systems are already demonstrating capabilities such as identifying skin cancer with greater accuracy than medications (Page 160). However, there are concerns about the potential impact of AI on employment, with advancements potentially leading to a decrease in white-collar jobs (Page 36).



Assistant:

AI-based systems can be purely software-based, acting in the virtual world (e.g. voice assistants, image analysis, software, search engines, speech and face recognition systems) or AI can be embedded in hardware devices (e.g. advanced robots, autonomous cars, drones or Internet of Things applications). (SOURCE 3, Page 2) Furthermore, AI needs vast amounts of data to be developed. Machine learning, a type of AI, 

In [20]:
print(context)

How to cite this book chapter: 
Rehak, R. 2021. The Language Labyrinth: Constructive Critique on the Terminology 
Used in the AI Discourse. In: Verdegem, P . (ed.) AI for Everyone? Critical Perspec-
tives. Pp. 87–102. London: University of Westminster Press. DOI: https://doi.org 
/10.16997/book55.f. License: CC-BY-NC-ND 4.0
CHAPTER 6
The Language Labyrinth: Constructive 
Critique on the Terminology Used  
in the AI Discourse
Rainer Rehak
Introduction
In the seventies of the last century, the British physicist and science fiction 
writer Arthur C. Clarke coined the phrase of any sufficiently advanced tech -
nology being indistinguishable from magic – understood here as mystical 
forces not accessible to reason or science. In his stories Clarke often described 
technical artefacts such as anti-gravity engines, ‘flowing’ roads or tiny atom-  
constructing machinery. In some of his stories, nobody knows exactly how

How to cite this book chapter: 
Akdag Salah, A. A. 2021. AI Bugs and Failu

Great Results! Splendid! 